# Reusable ML Pipeline with Scikit-learn

A modular, end-to-end machine learning pipeline that can be reused across different tabular datasets. It handles:

- Loading data
- Automatic detection of numeric vs categorical columns
- Preprocessing (imputation, scaling, encoding) via `ColumnTransformer`
- Model training via `Pipeline`
- Hyperparameter tuning via `GridSearchCV`
- Evaluation (classification or regression)
- Saving/loading the trained pipeline with `joblib`

Just swap the dataset and target column to reuse it on a new project.

## 1. Install & Import Libraries

In [ ]:
!pip install -q scikit-learn pandas numpy joblib matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    mean_squared_error, mean_absolute_error, r2_score
)

print('Libraries loaded successfully.')

## 2. Load Dataset

By default this loads the **Titanic** dataset from seaborn as a demo (since it's familiar from your previous notebooks). 

**To reuse on your own dataset:** replace this cell with `df = pd.read_csv('your_file.csv')` and set `TARGET_COLUMN` below.

In [ ]:
# --- OPTION A: Demo dataset (Titanic via seaborn) ---
df = sns.load_dataset('titanic')
TARGET_COLUMN = 'survived'
PROBLEM_TYPE = 'classification'  # 'classification' or 'regression'

# --- OPTION B: Use your own CSV instead ---
# df = pd.read_csv('/content/your_dataset.csv')
# TARGET_COLUMN = 'your_target_column'
# PROBLEM_TYPE = 'classification'  # or 'regression'

print(f'Shape: {df.shape}')
df.head()

## 3. Quick Exploration

In [ ]:
print('Info:')
df.info()
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\nTarget distribution:')
print(df[TARGET_COLUMN].value_counts())

## 4. Drop Unusable / High-Cardinality / Leakage Columns

Edit `COLUMNS_TO_DROP` per dataset (e.g. IDs, names, free text, columns that leak the target).

In [ ]:
# Edit this list for your dataset
COLUMNS_TO_DROP = ['deck', 'embark_town', 'alive', 'who', 'adult_male', 'class']
COLUMNS_TO_DROP = [c for c in COLUMNS_TO_DROP if c in df.columns]

df = df.drop(columns=COLUMNS_TO_DROP)
df = df.dropna(subset=[TARGET_COLUMN])  # rows without a target are useless

print(f'Dropped: {COLUMNS_TO_DROP}')
print(f'Remaining shape: {df.shape}')

## 5. Train/Test Split

In [ ]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

# Encode target if classification and non-numeric
if PROBLEM_TYPE == 'classification' and y.dtype == 'object':
    le = LabelEncoder()
    y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y if PROBLEM_TYPE == 'classification' else None
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 6. Build the Reusable Preprocessing Pipeline

Automatically detects numeric vs categorical columns and applies the right transforms. This part doesn't need to change between projects.

In [ ]:
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print(f'Numeric columns: {numeric_cols}')
print(f'Categorical columns: {categorical_cols}')

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

print('\nPreprocessor built.')

## 7. Build the Full Model Pipeline

Combines preprocessing + model into a single `Pipeline` object. Swap `model` for any estimator you like.

In [ ]:
if PROBLEM_TYPE == 'classification':
    model = RandomForestClassifier(random_state=42)
else:
    model = RandomForestRegressor(random_state=42)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

pipeline

## 8. Hyperparameter Tuning with GridSearchCV

Edit `param_grid` keys to match whichever model you place in the pipeline above (prefix with `model__`).

In [ ]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 5, 10],
    'model__min_samples_split': [2, 5]
}

scoring = 'accuracy' if PROBLEM_TYPE == 'classification' else 'r2'

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=5,
    scoring=scoring,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'\nBest params: {grid_search.best_params_}')
print(f'Best CV score: {grid_search.best_score_:.4f}')

best_pipeline = grid_search.best_estimator_

## 9. Evaluate on Test Set

In [ ]:
y_pred = best_pipeline.predict(X_test)

if PROBLEM_TYPE == 'classification':
    print('Accuracy: ', accuracy_score(y_test, y_pred))
    print('Precision:', precision_score(y_test, y_pred, average='weighted'))
    print('Recall:   ', recall_score(y_test, y_pred, average='weighted'))
    print('F1 Score: ', f1_score(y_test, y_pred, average='weighted'))
    print('\nClassification Report:\n', classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()
else:
    print('MAE: ', mean_absolute_error(y_test, y_pred))
    print('MSE: ', mean_squared_error(y_test, y_pred))
    print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred)))
    print('R2:  ', r2_score(y_test, y_pred))

    plt.figure(figsize=(5, 4))
    plt.scatter(y_test, y_pred, alpha=0.6)
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Actual vs Predicted')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    plt.show()

## 10. Feature Importance (if supported by model)

In [ ]:
try:
    feature_names = best_pipeline.named_steps['preprocessor'].get_feature_names_out()
    importances = best_pipeline.named_steps['model'].feature_importances_

    feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(15)

    plt.figure(figsize=(8, 6))
    feat_imp.plot(kind='barh')
    plt.title('Top 15 Feature Importances')
    plt.gca().invert_yaxis()
    plt.show()
except AttributeError:
    print('This model does not expose feature_importances_.')

## 11. Save the Reusable Pipeline

The entire pipeline (preprocessing + tuned model) is saved as one object. Load it later and call `.predict()` directly on raw data — no need to repeat preprocessing manually.

In [ ]:
joblib.dump(best_pipeline, 'ml_pipeline.joblib')
print('Pipeline saved as ml_pipeline.joblib')

# Download it locally if on Colab
try:
    from google.colab import files
    files.download('ml_pipeline.joblib')
except ImportError:
    pass

## 12. Load & Reuse the Pipeline on New Data

This is the payoff: load the saved pipeline anywhere and predict on fresh raw rows without rewriting any preprocessing code.

In [ ]:
loaded_pipeline = joblib.load('ml_pipeline.joblib')

# Predict on a few rows from the test set as a sanity check
sample = X_test.iloc[:5]
predictions = loaded_pipeline.predict(sample)
print('Sample predictions:', predictions)

## How to Reuse This Notebook for a New Project

1. **Section 2**: load your own CSV and set `TARGET_COLUMN` + `PROBLEM_TYPE`.
2. **Section 4**: update `COLUMNS_TO_DROP` for IDs/leaky columns specific to your data.
3. **Section 7**: swap `model` for `LogisticRegression()`, `SVC()`, `LinearRegression()`, etc.
4. **Section 8**: update `param_grid` keys/values to match the new model's hyperparameters.
5. Run all cells — sections 6, 9, 10, 11, 12 require no changes since they auto-detect column types and adapt to classification vs regression.